In [11]:
import duckdb

con = duckdb.connect()

# Load SQLite extension
con.execute("INSTALL sqlite;")
con.execute("LOAD sqlite;")

# Attach source databases
con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_2t.sqlite' AS t_db
""")

con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_2d.sqlite' AS td_db
""")

con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_rh.sqlite' AS rh_db
""")

con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_vpd.sqlite' AS vpd_db
""")

con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_tp.sqlite' AS tp_db
""")

con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5MonthlyMeansFinal.sqlite' AS out_db
""")

BinderException: Binder Error: Unique file handle conflict: Cannot attach "rh_db" - the database file "/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_rh.sqlite" is already attached by database "rh_db"

In [ ]:
rhCut = 20

con.execute(f"""
CREATE OR REPLACE TABLE out_db.monthly_means AS            
SELECT
    t.point_id,

    SUBSTR(t.datetime, 1, 6) AS year_month,

    -- T mean
    SUM(
        t."0" + t."3" + t."6" + t."9" +
        t."12" + t."15" + t."18" + t."21"
    ) / (COUNT(*) * 8) AS t_mean,

    -- Td mean
    SUM(
        td."0" + td."3" + td."6" + td."9" +
        td."12" + td."15" + td."18" + td."21"
    ) / (COUNT(*) * 8) AS td_mean,

    -- RH mean
    SUM(
        rh."0" + rh."3" + rh."6" + rh."9" +
        rh."12" + rh."15" + rh."18" + rh."21"
    ) / (COUNT(*) * 8) AS rh_mean,

    -- VPD mean
    SUM(
        vpd."0" + vpd."3" + vpd."6" + vpd."9" +
        vpd."12" + vpd."15" + vpd."18" + vpd."21"
    ) / (COUNT(*) * 8) AS vpd_mean,

    -- TP monthly sum (NO DIVISION)
    SUM(
        CASE 
            WHEN tp."0" >= 0.000254 
            THEN tp."0" 
            ELSE 0 
        END
    ) AS tp_sum,

    -- RH sums
    SUM(
        CASE
            WHEN (
                rh."0"  < {rhCut} OR
                rh."3"  < {rhCut} OR
                rh."6"  < {rhCut} OR
                rh."9"  < {rhCut} OR
                rh."12" < {rhCut} OR
                rh."15" < {rhCut} OR
                rh."18" < {rhCut} OR
                rh."21" < {rhCut}
            )
            THEN 1 ELSE 0
        END
    ) AS days_with_rh_lt_{rhCut},

    (SUM(
        CASE WHEN rh."0"  < {rhCut} THEN 1 ELSE 0 END +
        CASE WHEN rh."3"  < {rhCut} THEN 1 ELSE 0 END +
        CASE WHEN rh."6"  < {rhCut} THEN 1 ELSE 0 END +
        CASE WHEN rh."9"  < {rhCut} THEN 1 ELSE 0 END +
        CASE WHEN rh."12" < {rhCut} THEN 1 ELSE 0 END +
        CASE WHEN rh."15" < {rhCut} THEN 1 ELSE 0 END +
        CASE WHEN rh."18" < {rhCut} THEN 1 ELSE 0 END +
        CASE WHEN rh."21" < {rhCut} THEN 1 ELSE 0 END
    )
    /
    (COUNT(*) * 8)
    ) AS rh_lt_{rhCut}_pct,

    (SUM(
        CASE WHEN vpd."0"  > 3.0 THEN 1 ELSE 0 END +
        CASE WHEN vpd."3"  > 3.0 THEN 1 ELSE 0 END +
        CASE WHEN vpd."6"  > 3.0 THEN 1 ELSE 0 END +
        CASE WHEN vpd."9"  > 3.0 THEN 1 ELSE 0 END +
        CASE WHEN vpd."12" > 3.0 THEN 1 ELSE 0 END +
        CASE WHEN vpd."15" > 3.0 THEN 1 ELSE 0 END +
        CASE WHEN vpd."18" > 3.0 THEN 1 ELSE 0 END +
        CASE WHEN vpd."21" > 3.0 THEN 1 ELSE 0 END
    )
    /
    (COUNT(*) * 8)
    ) AS vpd_gt_3_pct,

    -- VPD sums
    SUM(
        CASE
            WHEN (
                vpd."0"  > 3.0 OR
                vpd."3"  > 3.0 OR
                vpd."6"  > 3.0 OR
                vpd."9"  > 3.0 OR
                vpd."12" > 3.0 OR
                vpd."15" > 3.0 OR
                vpd."18" > 3.0 OR
                vpd."21" > 3.0
            )
            THEN 1 ELSE 0
        END
    ) AS days_with_vpd_gt_3

FROM t_db.daily_data t

JOIN td_db.daily_data td
    ON t.datetime = td.datetime
    AND t.point_id = td.point_id

JOIN rh_db.daily_data rh
    ON t.datetime = rh.datetime
    AND t.point_id = rh.point_id

JOIN vpd_db.daily_data vpd
    ON t.datetime = vpd.datetime
    AND t.point_id = vpd.point_id

JOIN tp_db.daily_data tp
    ON t.datetime = tp.datetime
    AND t.point_id = tp.point_id

GROUP BY
    t.point_id,
    year_month

ORDER BY
    t.point_id,
    year_month
""")


con.execute("""
CREATE INDEX idx_monthly_means  
ON out_db.monthly_means(point_id, year_month)
""")

con.close()

In [ ]:
con.execute("""
SHOW TABLES FROM out_db
""").df()

,name
0,monthly_means


In [1]:
import duckdb

con = duckdb.connect()

# Load SQLite extension
con.execute("INSTALL sqlite;")
con.execute("LOAD sqlite;")



con.execute("""
ATTACH '/home/joe/work/Fire/ML/Data/DB/era5MonthlyMeansFinal.sqlite' AS out_db
""")

In [3]:
con.execute("""
SELECT name 
FROM main.sqlite_master 
WHERE type='table'
""").fetchall()

[('monthly_means',)]

In [8]:
import sqlite3
import pandas as pd

In [1]:
import duckdb

con = duckdb.connect("/home/joe/work/Fire/ML/Data/DB/era5MonthlyMeansFinal.sqlite")

df = con.execute("""
SELECT * FROM monthly_means
""").df()

con.close()

In [3]:
df.columns

Index(['point_id', 'year_month', 't_mean', 'td_mean', 'rh_mean', 'vpd_mean',
       'tp_sum', 'days_with_rh_lt_20', 'rh_lt_20_pct', 'vpd_gt_3_pct',
       'days_with_vpd_gt_3'],
      dtype='object')